# AI-Powered Energy Consumption Forecasting: Exploratory Data Analysis

This notebook provides interactive exploration of the **Individual Household Electric Power Consumption** dataset from the UCI Machine Learning Repository.

### Objectives:
1. Inspect raw minute-level measurements and data coverage.
2. Verify defensible integration of active power (kW) into hourly energy (kWh).
3. Analyze diurnal (hourly) and seasonal (monthly) consumption dynamics.
4. Assess autocorrelation and justify lag selection ($t-1, t-2, t-3, t-24, t-168$).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# Paths
PROCESSED_PATH = Path('../data/processed/hourly_energy_consumption.csv')
if not PROCESSED_PATH.exists():
    PROCESSED_PATH = Path('data/processed/hourly_energy_consumption.csv')

## 1. Load Processed Hourly Energy Data

In [ ]:
df = pd.read_csv(PROCESSED_PATH, index_col=0, parse_dates=True)
print(f'Total hourly records: {len(df):,}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
df.head()

## 2. Statistical Summary of Hourly Consumption (kWh)

In [ ]:
df[['energy_kwh', 'mean_power_kw', 'valid_minutes_count']].describe()

## 3. Diurnal (Hourly) Consumption Pattern

In [ ]:
df['hour'] = df.index.hour
hourly_stats = df.groupby('hour')['energy_kwh'].agg(['mean', 'std'])

plt.figure(figsize=(10, 5))
plt.plot(hourly_stats.index, hourly_stats['mean'], marker='o', color='#0288d1', linewidth=2.5, label='Mean Consumption')
plt.fill_between(hourly_stats.index, np.maximum(0, hourly_stats['mean'] - hourly_stats['std']*0.5),
                 hourly_stats['mean'] + hourly_stats['std']*0.5, color='#0288d1', alpha=0.2, label='±0.5 Std Dev')
plt.title('Household Diurnal Consumption Profile', fontweight='bold')
plt.xlabel('Hour of Day (0–23)')
plt.ylabel('Energy Consumption (kWh)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.show()

## 4. Weekday vs Weekend Comparison

In [ ]:
df['is_weekend'] = df.index.dayofweek >= 5
weekday_mean = df[~df['is_weekend']].groupby('hour')['energy_kwh'].mean()
weekend_mean = df[df['is_weekend']].groupby('hour')['energy_kwh'].mean()

plt.figure(figsize=(10, 5))
plt.plot(weekday_mean.index, weekday_mean.values, label='Weekday (Mon-Fri)', color='#388e3c', linewidth=2, marker='s')
plt.plot(weekend_mean.index, weekend_mean.values, label='Weekend (Sat-Sun)', color='#e64a19', linewidth=2, marker='^')
plt.title('Electricity Usage Profile: Weekday vs Weekend', fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Mean Energy (kWh)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.show()

## 5. Autocorrelation Analysis & Justification of Lags

Evaluating autocorrelation confirms strong correlation at lag 1 (immediate persistence), lag 24 (daily cycle), and lag 168 (weekly cycle).

In [ ]:
lags_to_check = [1, 2, 3, 6, 12, 24, 48, 72, 168]
autocorr_vals = [df['energy_kwh'].autocorr(lag=lag) for lag in lags_to_check]

plt.figure(figsize=(10, 5))
plt.bar([str(l) for l in lags_to_check], autocorr_vals, color='#5c6bc0', edgecolor='black')
plt.title('Autocorrelation of Hourly Consumption at Key Lag Horizons', fontweight='bold')
plt.xlabel('Lag Horizon (Hours)')
plt.ylabel('Autocorrelation Coefficient')
plt.axhline(0, color='black', linestyle='--')
for i, v in enumerate(autocorr_vals):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
plt.show()